# This Document Has all The Analysis of the Final Paper

## Part 1: Checking missing classes during the prediction, comparison with dummy classifier

In [2]:
import os
import re
import pandas as pd
from sklearn.metrics import f1_score
from collections import Counter

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def analyze_predictions_llms(hold_out_y_true: pd.Series, hold_out_y_pred: pd.Series,
                        num_examples: int, llm_model: str, dataset_name: str) -> pd.DataFrame():
    """
    Analyzes the performance of a classification model's predictions against true labels.

    This function compares the set of true labels with the set of predicted labels,
    identifies missing and extra classes in the predictions, calculates the
    micro-averaged F1 score for a majority class predictor baseline, and
    calculates the micro-averaged F1 score for the actual model predictions.

    Args:
        hold_out_y_true (pd.Series): A pandas Series containing the true labels
                                     for the hold-out dataset.
        hold_out_y_pred (pd.Series): A pandas Series containing the predicted labels
                                     from the model for the hold-out dataset.
        num_examples (int): The number of examples in the dataset.
        llm_model (str): The name of the language model used for predictions.
        dataset_name (str): The name of the dataset.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the analysis results, including
                      dataset name, LLM model, number of examples, missing classes
                      in predictions, extra classes in predictions, micro F1 score
                      for the majority class predictor, and micro F1 score for the
                      actual predictions.
    """
    # Step 1: Convert to sets for analysis
    true_classes = set(hold_out_y_true)
    pred_classes = set(hold_out_y_pred)

    # Step 2: Missing and extra classes
    missing_in_pred = true_classes - pred_classes
    extra_in_pred = pred_classes - true_classes

    # # Step 3: Print results
    # print("Dataset:", dataset_name)
    # print("LLM Model:", llm_model)
    # print("Number of Examples:", num_examples)
    # print("All true labels found in predictions:", missing_in_pred == set())
    # print("Missing classes in predictions:", missing_in_pred)
    # print("Extra classes in predictions:", extra_in_pred)

    if len(pred_classes) == 1:
        only_one_predicted_label = True
    else:
        only_one_predicted_label = False

    # Step 4: Majority class in ground truth
    majority_class = hold_out_y_true.mode().iloc[0]

    # Step 5: Predicting only the majority class
    pred_majority_only = [majority_class] * len(hold_out_y_true)

    # Step 6: F1-Score (Micro)
    f1_micro_majority = f1_score(hold_out_y_true, pred_majority_only, average='micro')
    print("F1 Score (Micro) for Majority Class Predictor:", f1_micro_majority)

    # Step 7: Actual prediction performance
    f1_micro_actual = f1_score(hold_out_y_true, hold_out_y_pred, average='micro')
    print("F1 Score (Micro) for Actual Predictions:", f1_micro_actual)

    return pd.DataFrame({
        "predicted_dataset": dataset_name,
        "selected_llm": llm_model,
        "number_of_examples": num_examples,
        "missing_in_pred": [missing_in_pred],
        "extra_in_pred": [extra_in_pred],
        "only_one_predicted_label": [only_one_predicted_label],
        "f1_micro_majority": f1_micro_majority,
        "hold_out_f1_score": f1_micro_actual,
        "y_true_labels": [list(true_classes)],
        "y_pred_labels": [list(pred_classes)]
    })

### Llama

In [ ]:
llama_output_results_df = pd.DataFrame()


results_llama_5_shot_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/llama/five_shot/")
results_llama_5_shot_list = [f"/content/drive/MyDrive/Colab Notebooks/output/llama/five_shot/{item}" for item in results_llama_5_shot_list]
results_llama_0_shot_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/llama/zero_shot/")
results_llama_0_shot_list = [f"/content/drive/MyDrive/Colab Notebooks/output/llama/zero_shot/{item}" for item in results_llama_0_shot_list]
results_llama_list = results_llama_5_shot_list + results_llama_0_shot_list

for file_result in results_llama_list:
    df = pd.read_csv(file_result, sep=";")
    df["hold_out_y_true"] = df["hold_out_y_true"].astype(str)
    df["hold_out_y_pred"] = df["hold_out_y_pred"].astype(str)
    output_df = analyze_predictions_llms(hold_out_y_true=df["hold_out_y_true"],
                                    hold_out_y_pred=df["hold_out_y_pred"],
                                    num_examples=df["number_of_examples"].iloc[0],
                                    llm_model=df["selected_llm"].iloc[0],
                                    dataset_name=df["name_dataset"].iloc[0]
                                    )
    llama_output_results_df = pd.concat([llama_output_results_df, output_df])
llama_output_results_df = llama_output_results_df.sort_values(by="predicted_dataset")
llama_output_results_df = llama_output_results_df.reset_index(drop=True)

F1 Score (Micro) for Majority Class Predictor: 0.3070539419087137
F1 Score (Micro) for Actual Predictions: 0.44813278008298757
F1 Score (Micro) for Majority Class Predictor: 0.29486706056129985
F1 Score (Micro) for Actual Predictions: 0.17402141802067947
F1 Score (Micro) for Majority Class Predictor: 0.904983865184654
F1 Score (Micro) for Actual Predictions: 0.49372534958766584
F1 Score (Micro) for Majority Class Predictor: 0.1986830804466075
F1 Score (Micro) for Actual Predictions: 0.1760664185513885
F1 Score (Micro) for Majority Class Predictor: 0.20062329567588624
F1 Score (Micro) for Actual Predictions: 0.1098558628749513
F1 Score (Micro) for Majority Class Predictor: 0.5208333333333334
F1 Score (Micro) for Actual Predictions: 0.6875
F1 Score (Micro) for Majority Class Predictor: 0.3416666666666667
F1 Score (Micro) for Actual Predictions: 0.2916666666666667
F1 Score (Micro) for Majority Class Predictor: 0.564875491480996
F1 Score (Micro) for Actual Predictions: 0.0982961992136304
F

In [ ]:
llama_output_results_df[llama_output_results_df["hold_out_f1_score"] < 0.15]

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
6,ed_04,LLAMA_3_2_3B_INSTRUCT,0,{},{---},False,0.904984,0.126210,"[not_hope, other, hope]","[not_hope, ---, other, hope]"
10,fn_01,LLAMA_3_2_3B_INSTRUCT,0,{barely_true},{---},False,0.200623,0.134788,"[half_true, false, mostly_true, true, barely_t...","[---, half_true, false, mostly_true, true, pan..."
11,fn_01,LLAMA_3_2_3B_INSTRUCT,5,{barely_true},{},False,0.200623,0.109856,"[half_true, false, mostly_true, true, barely_t...","[half_true, false, mostly_true, true, pants_on..."
16,fn_04,LLAMA_3_2_3B_INSTRUCT,0,"{misattributed, legend, unproven, outdated}",{---},False,0.564875,0.113368,"[misattributed, mostly_false, false_and_true, ...","[---, mostly_false, false_and_true, false, mos..."
17,fn_04,LLAMA_3_2_3B_INSTRUCT,5,"{legend, unproven}",{---},False,0.564875,0.098296,"[misattributed, mostly_false, false_and_true, ...","[correct_attribution, misattributed, ---, most..."
24,fn_08,LLAMA_3_2_3B_INSTRUCT,0,{misleading},{---},False,0.843008,0.047493,"[partly_false, mostly_false, false, misleading...","[---, partly_false, mostly_false, false, true]"
25,fn_08,LLAMA_3_2_3B_INSTRUCT,5,{},{},False,0.843008,0.091029,"[partly_false, mostly_false, false, misleading...","[partly_false, mostly_false, false, misleading..."
58,hs_15,LLAMA_3_2_3B_INSTRUCT,5,{},"{##noise, ---, #noise, homophobic}",False,0.604955,0.107691,"[appropriative, noise, homonym, derogatory, no...","[##noise, ---, appropriative, homophobic, nois..."
74,pc_08,LLAMA_3_2_3B_INSTRUCT,5,{},{},False,0.479628,0.142244,"[positive, negative, neutral]","[positive, negative, neutral]"
87,ua_02,LLAMA_3_2_3B_INSTRUCT,0,{},"{---, 40, 50, 20}",False,0.221352,0.127326,"[9, 10, 5, 3, 2, 4, 8, 1, 6, 7]","[---, 5, 10, 9, 3, 20, 50, 2, 4, 40, 8, 1, 6, 7]"


### Mistral

In [ ]:
mistral_output_results_df = pd.DataFrame()

results_mistral_5_shot_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/mistral/five_shot/")
results_mistral_5_shot_list = [f"/content/drive/MyDrive/Colab Notebooks/output/mistral/five_shot/{item}" for item in results_mistral_5_shot_list]
results_mistral_0_shot_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/mistral/zero_shot/")
results_mistral_0_shot_list = [f"/content/drive/MyDrive/Colab Notebooks/output/mistral/zero_shot/{item}" for item in results_mistral_0_shot_list]
results_mistral_list = results_mistral_5_shot_list + results_mistral_0_shot_list

for file_result in results_mistral_list:
    df = pd.read_csv(file_result, sep=";")
    df["hold_out_y_true"] = df["hold_out_y_true"].astype(str)
    df["hold_out_y_pred"] = df["hold_out_y_pred"].astype(str)
    output_df = analyze_predictions_llms(hold_out_y_true=df["hold_out_y_true"],
                                    hold_out_y_pred=df["hold_out_y_pred"],
                                    num_examples=df["number_of_examples"].iloc[0],
                                    llm_model=df["selected_llm"].iloc[0],
                                    dataset_name=df["name_dataset"].iloc[0]
                                    )
    mistral_output_results_df = pd.concat([mistral_output_results_df, output_df])
mistral_output_results_df = mistral_output_results_df.sort_values(by="predicted_dataset")
mistral_output_results_df = mistral_output_results_df.reset_index(drop=True)

F1 Score (Micro) for Majority Class Predictor: 0.3428708248533461
F1 Score (Micro) for Actual Predictions: 0.5773118297959672
F1 Score (Micro) for Majority Class Predictor: 0.5004814588202253
F1 Score (Micro) for Actual Predictions: 0.8898813404615986
F1 Score (Micro) for Majority Class Predictor: 0.5583801505840487
F1 Score (Micro) for Actual Predictions: 0.324315864045999
F1 Score (Micro) for Majority Class Predictor: 0.9060041609778298
F1 Score (Micro) for Actual Predictions: 0.42300565632923737
F1 Score (Micro) for Majority Class Predictor: 0.904983865184654
F1 Score (Micro) for Actual Predictions: 0.19254212979562568
F1 Score (Micro) for Majority Class Predictor: 0.29486706056129985
F1 Score (Micro) for Actual Predictions: 0.21399556868537667
F1 Score (Micro) for Majority Class Predictor: 0.3416666666666667
F1 Score (Micro) for Actual Predictions: 0.38333333333333336
F1 Score (Micro) for Majority Class Predictor: 0.3070539419087137
F1 Score (Micro) for Actual Predictions: 0.543568

In [ ]:
mistral_output_results_df[mistral_output_results_df["hold_out_f1_score"] < 0.15]

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
74,pc_08,MISTRAL_NEMO_INSTRUCT_2407,5,{},"{nafta, moscow, guilty, sexual-harassment, clo...",False,0.479628,0.078628,"[positive, negative, neutral]","[nafta, moscow, guilty, sexual-harassment, clo..."
87,ua_02,MISTRAL_NEMO_INSTRUCT_2407,0,{2},{30},False,0.221352,0.121450,"[9, 10, 5, 3, 2, 4, 8, 1, 6, 7]","[30, 9, 5, 10, 3, 4, 8, 1, 6, 7]"
88,ua_03,MISTRAL_NEMO_INSTRUCT_2407,5,{},"{width, green, voltage, recommended, refurbish...",False,0.666016,0.126504,"[5, 10, 3, 1, 7]","[width, green, voltage, price, gls, poor, perf..."
92,ua_05,MISTRAL_NEMO_INSTRUCT_2407,0,{},{},False,0.776119,0.129353,"[10, 5, 3, 1, 7]","[5, 10, 3, 1, 7]"
93,ua_05,MISTRAL_NEMO_INSTRUCT_2407,5,{},{},False,0.776119,0.124378,"[10, 5, 3, 1, 7]","[5, 10, 3, 1, 7]"
95,ua_06,MISTRAL_NEMO_INSTRUCT_2407,5,{},"{good, ---, 4}",False,0.442095,0.128985,"[5, 10, 3, 1, 7]","[good, ---, 5, 10, 3, 4, 1, 7]"


### AutoML - Sampled

In [ ]:
def analyze_predictions_automl_sampled(hold_out_y_true: pd.Series,
                                       hold_out_y_pred: pd.Series,
                                       selected_system: str,
                                       dataset_name: str,
                                       method: str,
                                       percentage: int) -> pd.DataFrame():
    """
    Analyzes the performance of a classification model's predictions against true labels.

    This function compares the set of true labels with the set of predicted labels,
    identifies missing and extra classes in the predictions, calculates the
    micro-averaged F1 score for a majority class predictor baseline, and
    calculates the micro-averaged F1 score for the actual model predictions.

    Args:
        hold_out_y_true (pd.Series): A pandas Series containing the true labels
                                     for the hold-out dataset.
        hold_out_y_pred (pd.Series): A pandas Series containing the predicted labels
                                     from the model for the hold-out dataset.
        # The following args are scalars for the result DataFrame row
        selected_system (str): The name of the AutoML system used.
        dataset_name (str): The name of the dataset.
        method (str): The sampling method used.
        percentage (int): The sampling percentage.


    Returns:
        pd.DataFrame: A pandas DataFrame containing the analysis results, including
                      dataset name, AutoML system, method, percentage, missing classes
                      in predictions, extra classes in predictions, micro F1 score
                      for the majority class predictor, micro F1 score for the
                      actual predictions, and the lists of unique true and predicted labels.
    """
    # Step 1: Convert to sets for analysis
    true_classes = set(hold_out_y_true)
    pred_classes = set(hold_out_y_pred)

    # Step 2: Missing and extra classes
    missing_in_pred = list(true_classes - pred_classes) # Convert sets to lists
    extra_in_pred = list(pred_classes - true_classes)   # Convert sets to lists

    # # Step 3: Print results
    # print("Dataset:", dataset_name)
    # print("System:", selected_system)
    # print("Sampling Method:", method)
    # print("Percentage", percentage)
    # print("All true labels found in predictions:", missing_in_pred == []) # Check if the list is empty
    # print("Missing classes in predictions:", missing_in_pred)
    # print("Extra classes in predictions:", extra_in_pred)

    # Step 4: Majority class in ground truth
    # Handle potential case of empty input series
    if not hold_out_y_true.empty:
        majority_class = hold_out_y_true.mode().iloc[0]
    else:
        majority_class = None # Or some other indicator for no majority class


    # Step 5: Predicting only the majority class
    # Only create prediction list if there are true labels
    if majority_class is not None:
        pred_majority_only = [majority_class] * len(hold_out_y_true)
    else:
        pred_majority_only = []

    if len(pred_classes) == 1:
        only_one_predicted_label = True
    else:
        only_one_predicted_label = False

    # Step 6: F1-Score (Micro)
    # Only calculate F1 scores if there are true labels and predictions
    if len(hold_out_y_true) > 0 and len(hold_out_y_pred) > 0:
        try:
            f1_micro_majority = f1_score(hold_out_y_true, pred_majority_only, average='micro')
            f1_micro_actual = f1_score(hold_out_y_true, hold_out_y_pred, average='micro')
        except ValueError as e:
             # Handle cases where f1_score might still fail (e.g., different labels in y_true and y_pred)
            print(f"Error calculating F1-score: {e}")
            f1_micro_majority = float('nan')
            f1_micro_actual = float('nan')
    else:
        f1_micro_majority = float('nan')
        f1_micro_actual = float('nan')


    # print("F1 Score (Micro) for Majority Class Predictor:", f1_micro_majority)
    # print("F1 Score (Micro) for Actual Predictions:", f1_micro_actual)

    # Wrap lists in another list to ensure consistent length (1) for DataFrame construction
    return pd.DataFrame({
        "predicted_dataset": [dataset_name],
        "automl_system": [selected_system],
        "method": [method],
        "percentage": [percentage],
        "missing_in_pred": [missing_in_pred],  # Wrap the list in another list
        "extra_in_pred": [extra_in_pred],      # Wrap the list in another list
        "only_one_predicted_label": [only_one_predicted_label],
        "f1_micro_majority": [f1_micro_majority],
        "hold_out_f1_score": [f1_micro_actual],
        "y_true_labels": [list(true_classes)],  # Wrap the list in another list
        "y_pred_labels": [list(pred_classes)]   # Wrap the list in another list
    })

In [ ]:
automl_sampled_df = pd.DataFrame()
results_automl_sampled_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/automl_sampled/")
results_automl_sampled_list = [f"/content/drive/MyDrive/Colab Notebooks/output/automl_sampled/{item}" for item in results_automl_sampled_list]

for file_path_name in results_automl_sampled_list:
    df = pd.read_csv(file_path_name, sep=";")
    label_str = df["hold_out_y_true"].iloc[0]
    processed_str = re.sub("\'|\[|\]", "", label_str)
    hold_out_y_true = processed_str.split(", ")
    label_str = df["hold_out_y_pred"].iloc[0]
    processed_str = re.sub("\'|\[|\]", "", label_str)
    hold_out_y_pred = processed_str.split(", ")
    # Create the DataFrame with each element as a row
    normalized_schema_df = pd.DataFrame({
        "selected_system": df["automl_system"].iloc[0],
        "predicted_dataset": df["dataset"].iloc[0],
        "method": df["method"].iloc[0],
        "percentage": df["percentage"].iloc[0],
        "hold_out_y_true": hold_out_y_true,
        "hold_out_y_pred": hold_out_y_pred
    })
    output_df = analyze_predictions_automl_sampled(hold_out_y_true = normalized_schema_df["hold_out_y_true"],
                                       hold_out_y_pred = normalized_schema_df["hold_out_y_pred"],
                                       selected_system = normalized_schema_df["selected_system"].iloc[0],
                                       dataset_name = normalized_schema_df["predicted_dataset"].iloc[0],
                                       method = normalized_schema_df["method"].iloc[0],
                                       percentage = normalized_schema_df["percentage"].iloc[0])
    automl_sampled_df = pd.concat([automl_sampled_df, output_df])
automl_sampled_df = automl_sampled_df.sort_values(by=["predicted_dataset","automl_system"])
automl_sampled_df = automl_sampled_df.reset_index(drop=True)
automl_sampled_df

,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_01_multi,AutoGluon,gmm_random,50,[],[],False,0.307054,0.443983,"[sent_06, sent_05, sent_01, sent_02, sent_04, ...","[sent_06, sent_05, sent_01, sent_02, sent_04, ..."
1,ed_01_multi,AutoGluon,gmm_random,25,[],[],False,0.307054,0.385892,"[sent_06, sent_05, sent_01, sent_02, sent_04, ...","[sent_06, sent_05, sent_01, sent_02, sent_04, ..."
2,ed_01_multi,AutoGluon,gmm_random,75,[],[],False,0.307054,0.448133,"[sent_06, sent_05, sent_01, sent_02, sent_04, ...","[sent_06, sent_05, sent_01, sent_02, sent_04, ..."
3,ed_01_multi,FLAML,gmm_random,25,[],[],False,0.307054,0.427386,"[sent_06, sent_05, sent_01, sent_02, sent_04, ...","[sent_06, sent_05, sent_01, sent_02, sent_04, ..."
4,ed_01_multi,FLAML,gmm_random,50,[],[],False,0.307054,0.419087,"[sent_06, sent_05, sent_01, sent_02, sent_04, ...","[sent_06, sent_05, sent_01, sent_02, sent_04, ..."
...,...,...,...,...,...,...,...,...,...,...,...
436,ua_06_multi,FLAML,gmm_random,50,[],[],False,0.442095,0.531230,"[5, 3, 2, 4, 1]","[5, 3, 2, 4, 1]"
437,ua_06_multi,FLAML,gmm_random,25,[],[],False,0.442095,0.506831,"[5, 3, 2, 4, 1]","[5, 3, 2, 4, 1]"
438,ua_06_multi,Random Forest with Random Search,gmm_random,75,"[1, 2, 4, 3]",[],True,0.442095,0.442095,"[5, 3, 2, 4, 1]",[5]
439,ua_06_multi,Random Forest with Random Search,gmm_random,25,"[1, 2, 4, 3]",[],True,0.442095,0.442095,"[5, 3, 2, 4, 1]",[5]


In [ ]:
automl_sampled_df[automl_sampled_df["hold_out_f1_score"] < 0.15]

,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels


### AutoML - HPO

In [3]:
def analyze_predictions_automl_hpo(hold_out_y_true: list,
                                    hold_out_y_pred: list,
                                    selected_system: str,
                                    method: str,
                                    percentage: int,
                                    trained_dataset: str,
                                    predicted_dataset: str,
                                    automl_system: str,
                                    ) -> pd.DataFrame():
    """
    Analyzes the performance of a classification model's predictions against true labels.

    This function compares the set of true labels with the set of predicted labels,
    identifies missing and extra classes in the predictions, calculates the
    micro-averaged F1 score for a majority class predictor baseline, and
    calculates the micro-averaged F1 score for the actual model predictions.

    Args:
        hold_out_y_true (list): A pandas Series containing the true labels
                                     for the hold-out dataset.
        hold_out_y_pred (list): A pandas Series containing the predicted labels
                                     from the model for the hold-out dataset.
        selected_system (str): The name of the AutoML system used.
        method (str): The sampling method used.
        percentage (int): The sampling percentage.
        trained_dataset (str): the trained dataset name in the output DataFrame.
        predicted_dataset (str): the predicted dataset name in the output DataFrame.
        automl_system (str): the AutoML system name in the output DataFrame.
        method (str): the method in the output DataFrame.
        percentage (str): the percentage in the output DataFrame.


    Returns:
        pd.DataFrame: A pandas DataFrame containing the analysis results, including
                      dataset name, AutoML system, method, percentage, missing classes
                      in predictions, extra classes in predictions, micro F1 score
                      for the majority class predictor, micro F1 score for the
                      actual predictions, and the lists of unique true and predicted labels.
    """
    # Step 1: Convert to sets for analysis
    true_classes = set(hold_out_y_true)
    pred_classes = set(hold_out_y_pred)


    # Step 2: Missing and extra classes
    missing_in_pred = list(true_classes - pred_classes) # Convert sets to lists
    extra_in_pred = list(pred_classes - true_classes)   # Convert sets to lists

    # # Step 3: Print results
    # print("Trained Dataset:", trained_dataset)
    # print("Predicted Dataset:", predicted_dataset)
    # print("System:", selected_system)
    # print("Sampling Method:", method)
    # print("Percentage", percentage)
    # print("All true labels found in predictions:", missing_in_pred == []) # Check if the list is empty
    # print("Missing classes in predictions:", missing_in_pred)
    # print("Extra classes in predictions:", extra_in_pred)

    # Step 4: Majority class in ground truth
    # Handle potential case of empty input series
    hold_out_y_true = pd.Series(hold_out_y_true)
    hold_out_y_pred = pd.Series(hold_out_y_pred)
    if not hold_out_y_true.empty:
        majority_class = hold_out_y_true.mode().iloc[0]
    else:
        majority_class = None # Or some other indicator for no majority class


    # Step 5: Predicting only the majority class
    # Only create prediction list if there are true labels
    if majority_class is not None:
        pred_majority_only = [majority_class] * len(hold_out_y_true)
    else:
        pred_majority_only = []


    # Step 6: F1-Score (Micro)
    # Only calculate F1 scores if there are true labels and predictions
    if len(hold_out_y_true) > 0 and len(hold_out_y_pred) > 0:
        try:
            f1_micro_majority = f1_score(hold_out_y_true, pred_majority_only, average='micro')
            f1_micro_actual = f1_score(hold_out_y_true, hold_out_y_pred, average='micro')
        except ValueError as e:
             # Handle cases where f1_score might still fail (e.g., different labels in y_true and y_pred)
            print(f"Error calculating F1-score: {e}")
            f1_micro_majority = float('nan')
            f1_micro_actual = float('nan')
    else:
        f1_micro_majority = float('nan')
        f1_micro_actual = float('nan')

    if len(pred_classes) == 1:
        only_one_predicted_label = True
    else:
        only_one_predicted_label = False

    # print("F1 Score (Micro) for Majority Class Predictor:", f1_micro_majority)
    # print("F1 Score (Micro) for Actual Predictions:", f1_micro_actual)

    # Wrap lists in another list to ensure consistent length (1) for DataFrame construction
    return pd.DataFrame({
        "trained_dataset": [trained_dataset],
        "predicted_dataset": [predicted_dataset], # Note: predicted_dataset was not a parameter, added for consistency
        "automl_system": [selected_system],
        "method": [method],
        "percentage": [percentage],
        "missing_in_pred": [missing_in_pred],  # Wrap the list in another list
        "extra_in_pred": [extra_in_pred],      # Wrap the list in another list
        "only_one_predicted_label": [only_one_predicted_label],
        "f1_micro_majority": [f1_micro_majority],
        "hold_out_f1_score": [f1_micro_actual],
        "y_true_labels": [list(true_classes)],  # Wrap the list in another list
        "y_pred_labels": [list(pred_classes)],   # Wrap the list in another list
    })

In [5]:
automl_hpo_df = pd.DataFrame()
results_automl_hpo_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/hpo/")
results_automl_hpo_list = [f"/content/drive/MyDrive/Colab Notebooks/output/hpo/{item}" for item in results_automl_hpo_list]


for file_name in results_automl_hpo_list:
    df = pd.read_csv(file_name, sep=";")
    length = len(df)
    for counter in range(length):
        row_series = df.iloc[counter]
        label_str = row_series["hold_out_y_true"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_true_list = processed_str.split(",")
        label_str = row_series["hold_out_y_pred"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_pred_list = processed_str.split(",")
        output_df = analyze_predictions_automl_hpo(hold_out_y_true = hold_out_y_true_list,
                                                    hold_out_y_pred = hold_out_y_pred_list,
                                                    selected_system = row_series["automl_system"],
                                                    method = row_series["method"],
                                                    percentage = row_series["percentage"],
                                                    trained_dataset = row_series["trained_dataset"],
                                                    predicted_dataset = row_series["dataset_compared"],
                                                    automl_system = row_series["automl_system"],
                                                    )
        automl_hpo_df = pd.concat([automl_hpo_df, output_df])

In [6]:
automl_hpo_df

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,pc_02,pc_01,FLAML,gmm_random,100,[],[],False,0.501181,0.706188,"[negative, positive]","[negative, positive]"
0,pc_02,pc_03,FLAML,gmm_random,100,[],[],False,0.501573,0.731290,"[negative, positive]","[negative, positive]"
0,pc_02,pc_04,FLAML,gmm_random,100,[],[],False,0.500481,0.777217,"[negative, positive]","[negative, positive]"
0,pc_02,pc_06,FLAML,gmm_random,100,[],[],False,0.500622,0.825439,"[negative, positive]","[negative, positive]"
0,pc_02,pc_02,FLAML,gmm_random,100,[],[],False,0.501416,0.796034,"[negative, positive]","[negative, positive]"
...,...,...,...,...,...,...,...,...,...,...,...,...
0,pc_08,pc_06,Random Forest with Random Search,gmm_random,100,[negative],[],True,0.500622,0.499378,"[negative, positive]",[positive]
0,pc_08,pc_07,Random Forest with Random Search,gmm_random,100,"[negative, neutral]",[],True,0.502058,0.078189,"[positive, negative, neutral]",[positive]
0,pc_08,pc_08,Random Forest with Random Search,gmm_random,100,"[neutral, negative]",[],True,0.479628,0.479628,"[neutral, negative, positive]",[positive]
0,hs_02,hs_13,Random Forest with Random Search,gmm_random,100,"[neither, hate]",[],True,0.816537,0.126615,"[hate, offensive, neither]",[offensive]


In [7]:
automl_hpo_df[automl_hpo_df["hold_out_f1_score"] < 0.15]

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_03,ed_01,FLAML,gmm_random,100,"[sadness, fear, disgust, surprise, anger, joy]","[neutral, curiosity, admiration]",False,0.307054,0.000000,"[sadness, fear, surprise, anger, joy, disgust]","[neutral, curiosity, admiration]"
0,ed_03,ed_02,FLAML,gmm_random,100,[],"[optimism, confusion, admiration, disappointme...",False,0.342871,0.031118,"[sadness, fear, surprise, anger, joy, love]","[confusion, admiration, disapproval, amusement..."
0,fn_01,fn_02,FLAML,gmm_random,100,[],"[pants_on_fire, barely_true, half_true, mostly...",False,0.520833,0.145833,"[false, true]","[pants_on_fire, true, half_true, mostly_true, ..."
0,fn_01,fn_06,FLAML,gmm_random,100,[true],"[pants_on_fire, barely_true, half_true, mostly...",False,0.535865,0.088608,"[false, true]","[pants_on_fire, half_true, mostly_true, false,..."
0,fn_03,fn_02,FLAML,gmm_random,100,[true],"[mostly_false, pants_on_fire, half_true, mostl...",False,0.520833,0.083333,"[false, true]","[mostly_false, pants_on_fire, half_true, mostl..."
0,fn_03,fn_06,FLAML,gmm_random,100,[true],[half_true],False,0.535865,0.042194,"[false, true]","[false, half_true]"
0,hs_08,hs_02,FLAML,gmm_random,100,"[offensive, neither]",[not_hate],False,0.773053,0.001080,"[hate, offensive, neither]","[not_hate, hate]"
0,hs_08,hs_13,FLAML,gmm_random,100,"[offensive, neither]",[not_hate],False,0.816537,0.000000,"[hate, offensive, neither]","[not_hate, hate]"
0,pc_08,pc_07,FLAML,gmm_random,100,[negative],[],False,0.502058,0.084362,"[positive, negative, neutral]","[neutral, positive]"
0,pc_09,pc_02,FLAML,gmm_random,100,[],[neutral],False,0.501416,0.065156,"[negative, positive]","[neutral, negative, positive]"


In [ ]:
    # return pd.DataFrame({
    #     "trained_dataset": [trained_dataset],
    #     "predicted_dataset": [predicted_dataset], # Note: predicted_dataset was not a parameter, added for consistency
    #     "automl_system": [selected_system],
    #     "method": [method],
    #     "percentage": [percentage],
    #     "missing_in_pred": [missing_in_pred],  # Wrap the list in another list
    #     "extra_in_pred": [extra_in_pred],      # Wrap the list in another list
    #     "only_one_predicted_label": [only_one_predicted_label],
    #     "f1_micro_majority": [f1_micro_majority],
    #     "hold_out_f1_score": [f1_micro_actual],
    #     "y_true_labels": [list(true_classes)],  # Wrap the list in another list
    #     "y_pred_labels": [list(pred_classes)]   # Wrap the list in another list
    # })


# automl_hpo_df["are_trained_pred_equal"] =  automl_hpo_df.apply(lambda row: row.loc["trained_dataset"] == row.loc["predicted_dataset"], axis=1)
# flaml_hpo_df = automl_hpo_df[automl_hpo_df["automl_system"]=="FLAML"]
# equal_flaml_hpo_df = flaml_hpo_df[flaml_hpo_df["are_trained_pred_equal"]]
# equal_flaml_hpo_df = equal_flaml_hpo_df.groupby(["predicted_dataset"]).agg({"hold_out_f1_score": "mean"})
# equal_flaml_hpo_df = equal_flaml_hpo_df.rename()
# diff_flaml_hpo_df = flaml_hpo_df[~(flaml_hpo_df["are_trained_pred_equal"])]
# ag_hpo_df = automl_hpo_df[automl_hpo_df["automl_system"]=="AutoGluon"]
# equal_ag_hpo_df = ag_hpo_df[ag_hpo_df["are_trained_pred_equal"]]
# diff_ag_hpo_df = ag_hpo_df[~(ag_hpo_df["are_trained_pred_equal"])]
# rf_hpo_df = automl_hpo_df[automl_hpo_df["automl_system"]=="Random Forest with Random Search"]
# equal_rf_hpo_df = rf_hpo_df[rf_hpo_df["are_trained_pred_equal"]]
# diff_rf_hpo_df = rf_hpo_df[~(rf_hpo_df["are_trained_pred_equal"])]



### AutoML - Concat

In [ ]:
def analyze_predictions_automl_concat(hold_out_y_true: list,
                                    hold_out_y_pred: list,
                                    selected_system: str,
                                    method: str,
                                    percentage: int,
                                    trained_dataset: str,
                                    concatenated_dataset: str,
                                    predicted_dataset: str,
                                    automl_system: str,
                                    ) -> pd.DataFrame():
    """
    Analyzes the performance of a classification model's predictions against true labels.

    This function compares the set of true labels with the set of predicted labels,
    identifies missing and extra classes in the predictions, calculates the
    micro-averaged F1 score for a majority class predictor baseline, and
    calculates the micro-averaged F1 score for the actual model predictions.

    Args:
        hold_out_y_true (list): A pandas Series containing the true labels
                                     for the hold-out dataset.
        hold_out_y_pred (list): A pandas Series containing the predicted labels
                                     from the model for the hold-out dataset.
        selected_system (str): The name of the AutoML system used.
        method (str): The sampling method used.
        percentage (int): The sampling percentage.
        trained_dataset (str): the trained dataset name in the output DataFrame.
        predicted_dataset (str): the predicted dataset name in the output DataFrame.
        automl_system (str): the AutoML system name in the output DataFrame.
        method (str): the method in the output DataFrame.
        percentage (str): the percentage in the output DataFrame.


    Returns:
        pd.DataFrame: A pandas DataFrame containing the analysis results, including
                      dataset name, AutoML system, method, percentage, missing classes
                      in predictions, extra classes in predictions, micro F1 score
                      for the majority class predictor, micro F1 score for the
                      actual predictions, and the lists of unique true and predicted labels.
    """
    # Step 1: Convert to sets for analysis
    true_classes = set(hold_out_y_true)
    pred_classes = set(hold_out_y_pred)

    # Step 2: Missing and extra classes
    missing_in_pred = list(true_classes - pred_classes) # Convert sets to lists
    extra_in_pred = list(pred_classes - true_classes)   # Convert sets to lists

    # # Step 3: Print results
    # print("Trained Dataset:", trained_dataset)
    # print("Predicted Dataset:", predicted_dataset)
    # print("System:", selected_system)
    # print("Sampling Method:", method)
    # print("Percentage", percentage)
    # print("All true labels found in predictions:", missing_in_pred == []) # Check if the list is empty
    # print("Missing classes in predictions:", missing_in_pred)
    # print("Extra classes in predictions:", extra_in_pred)

    # Step 4: Majority class in ground truth
    # Handle potential case of empty input series
    hold_out_y_true = pd.Series(hold_out_y_true)
    hold_out_y_pred = pd.Series(hold_out_y_pred)
    if not hold_out_y_true.empty:
        majority_class = hold_out_y_true.mode().iloc[0]
    else:
        majority_class = None # Or some other indicator for no majority class


    # Step 5: Predicting only the majority class
    # Only create prediction list if there are true labels
    if majority_class is not None:
        pred_majority_only = [majority_class] * len(hold_out_y_true)
    else:
        pred_majority_only = []


    # Step 6: F1-Score (Micro)
    # Only calculate F1 scores if there are true labels and predictions
    if len(hold_out_y_true) > 0 and len(hold_out_y_pred) > 0:
        try:
            f1_micro_majority = f1_score(hold_out_y_true, pred_majority_only, average='micro')
            f1_micro_actual = f1_score(hold_out_y_true, hold_out_y_pred, average='micro')
        except ValueError as e:
             # Handle cases where f1_score might still fail (e.g., different labels in y_true and y_pred)
            print(f"Error calculating F1-score: {e}")
            f1_micro_majority = float('nan')
            f1_micro_actual = float('nan')
    else:
        f1_micro_majority = float('nan')
        f1_micro_actual = float('nan')

    if len(pred_classes) == 1:
        only_one_predicted_label = True
    else:
        only_one_predicted_label = False

    # print("F1 Score (Micro) for Majority Class Predictor:", f1_micro_majority)
    # print("F1 Score (Micro) for Actual Predictions:", f1_micro_actual)
    print(f"{trained_dataset=}")
    print(f"{concatenated_dataset=}")
    print(f"{predicted_dataset=}\n")

    # Wrap lists in another list to ensure consistent length (1) for DataFrame construction
    return pd.DataFrame({
        "trained_dataset": [trained_dataset],
        "predicted_dataset": [predicted_dataset], # Note: predicted_dataset was not a parameter, added for consistency
        "concatenated_dataset": [concatenated_dataset],
        "automl_system": [selected_system],
        "method": [method],
        "percentage": [percentage],
        "missing_in_pred": [missing_in_pred],  # Wrap the list in another list
        "extra_in_pred": [extra_in_pred],      # Wrap the list in another list
        "only_one_predicted_label": [only_one_predicted_label],
        "f1_micro_majority": [f1_micro_majority],
        "hold_out_f1_score": [f1_micro_actual],
        "y_true_labels": [list(true_classes)],  # Wrap the list in another list
        "y_pred_labels": [list(pred_classes)]   # Wrap the list in another list
    })

#### AutoGluon

In [ ]:
automl_concat_ag_df = pd.DataFrame()
results_automl_concat_ag_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/concat/autogluon")
results_automl_concat_ag_list = [f"/content/drive/MyDrive/Colab Notebooks/output/concat/autogluon/{item}" for item in results_automl_concat_ag_list]
results_automl_concat_ag_list.sort()

for file_name in results_automl_concat_ag_list:
    df = pd.read_csv(file_name, sep=";")
    length = len(df)
    print(f"{file_name=}")
    for counter in range(length):
        row_series = df.iloc[counter]
        label_str = row_series["hold_out_y_true"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_true_list = processed_str.split(",")
        label_str = row_series["hold_out_y_pred"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_pred_list = processed_str.split(",")
        # print(row_series["trained_dataset"])
        print(str(row_series["combination_trained"]))

        print(f"trained dataset: {row_series['trained_dataset']}")
        print(f"concatenated dataset: {'_'.join(str(row_series['combination_trained']).split('_')[8:10])}")
        print(f"predicted dataset: {'_'.join(str(row_series['combination_trained']).split('_')[-2:])}")
        # break
        output_df = analyze_predictions_automl_concat(hold_out_y_true = hold_out_y_true_list,
                                                    hold_out_y_pred = hold_out_y_pred_list,
                                                    concatenated_dataset = '_'.join(str(row_series['combination_trained']).split('_')[8:10]),
                                                    selected_system = row_series["automl_system"],
                                                    method = row_series["method"],
                                                    percentage = row_series["percentage"],
                                                    trained_dataset = row_series["trained_dataset"],
                                                    predicted_dataset = "_".join(str(row_series["combination_trained"]).split("_")[-2:]),
                                                    automl_system = row_series["automl_system"]
                                                    )
        automl_concat_ag_df = pd.concat([automl_concat_ag_df, output_df])
    # break

Streaming output truncated to the last 5000 lines.
predicted_dataset='pc_03'

base_dataset_pc_01_dataset_sampled_to_concat_pc_02_percentage_75_holdout_dataset_to_be_predicted_pc_04
trained dataset: pc_01
concatenated dataset: pc_02
predicted dataset: pc_04
trained_dataset='pc_01'
concatenated_dataset='pc_02'
predicted_dataset='pc_04'

base_dataset_pc_01_dataset_sampled_to_concat_pc_02_percentage_75_holdout_dataset_to_be_predicted_pc_06
trained dataset: pc_01
concatenated dataset: pc_02
predicted dataset: pc_06
trained_dataset='pc_01'
concatenated_dataset='pc_02'
predicted_dataset='pc_06'

base_dataset_pc_01_dataset_sampled_to_concat_pc_02_percentage_75_holdout_dataset_to_be_predicted_pc_01
trained dataset: pc_01
concatenated dataset: pc_02
predicted dataset: pc_01
trained_dataset='pc_01'
concatenated_dataset='pc_02'
predicted_dataset='pc_01'

base_dataset_pc_01_dataset_sampled_to_concat_pc_03_percentage_25_holdout_dataset_to_be_predicted_pc_02
trained dataset: pc_01
concatenated datase

In [ ]:
string_ = "base_dataset_ed_03_dataset_sampled_to_concat_ed_01_percentage_25_holdout_dataset_to_be_predicted_ed_01"
"_".join(string_.split("_")[8:10])

In [ ]:
automl_concat_ag_df[automl_concat_ag_df["hold_out_f1_score"] < 0.15]

#### Random Forest

In [ ]:
automl_concat_rf_df = pd.DataFrame()
results_automl_concat_rf_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/concat/random_forest")
results_automl_concat_rf_list = [f"/content/drive/MyDrive/Colab Notebooks/output/concat/random_forest/{item}" for item in results_automl_concat_rf_list]


for file_name in results_automl_concat_rf_list:
    df = pd.read_csv(file_name, sep=";")
    length = len(df)
    print(f"{file_name=}")
    for counter in range(length):
        row_series = df.iloc[counter]
        label_str = row_series["hold_out_y_true"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_true_list = processed_str.split(",")
        label_str = row_series["hold_out_y_pred"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_pred_list = processed_str.split(",")
        # print(row_series["trained_dataset"])
        print(str(row_series["combination_trained"]))

        print(f"trained dataset: {row_series['trained_dataset']}")
        print(f"concatenated dataset: {'_'.join(str(row_series['combination_trained']).split('_')[8:10])}")
        print(f"predicted dataset: {'_'.join(str(row_series['combination_trained']).split('_')[-2:])}")
        # break
        output_df = analyze_predictions_automl_concat(hold_out_y_true = hold_out_y_true_list,
                                                    hold_out_y_pred = hold_out_y_pred_list,
                                                    concatenated_dataset = '_'.join(str(row_series['combination_trained']).split('_')[8:10]),
                                                    selected_system = row_series["automl_system"],
                                                    method = row_series["method"],
                                                    percentage = row_series["percentage"],
                                                    trained_dataset = row_series["trained_dataset"],
                                                    predicted_dataset = "_".join(str(row_series["combination_trained"]).split("_")[-2:]),
                                                    automl_system = row_series["automl_system"]
                                                    )
        automl_concat_rf_df = pd.concat([automl_concat_rf_df, output_df])

Streaming output truncated to the last 5000 lines.
trained dataset: pc_08
concatenated dataset: pc_07
predicted dataset: pc_03
trained_dataset='pc_08'
concatenated_dataset='pc_07'
predicted_dataset='pc_03'

base_dataset_pc_08_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_04
trained dataset: pc_08
concatenated dataset: pc_07
predicted dataset: pc_04
trained_dataset='pc_08'
concatenated_dataset='pc_07'
predicted_dataset='pc_04'

base_dataset_pc_08_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_05
trained dataset: pc_08
concatenated dataset: pc_07
predicted dataset: pc_05
trained_dataset='pc_08'
concatenated_dataset='pc_07'
predicted_dataset='pc_05'

base_dataset_pc_08_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_06
trained dataset: pc_08
concatenated dataset: pc_07
predicted dataset: pc_06
trained_dataset='pc_08'
concatenated_dataset='pc_07'
predicted_dataset='pc_06'

base_dataset_pc_

In [ ]:
automl_concat_rf_df[automl_concat_rf_df["hold_out_f1_score"] < 0.15]

#### FLAML

In [ ]:
automl_concat_flaml_df = pd.DataFrame()
results_automl_concat_flaml_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/concat/flaml")
results_automl_concat_flaml_list = [f"/content/drive/MyDrive/Colab Notebooks/output/concat/flaml/{item}" for item in results_automl_concat_flaml_list]


for file_name in results_automl_concat_flaml_list:
    df = pd.read_csv(file_name, sep=";")
    length = len(df)
    print(f"{file_name=}")
    for counter in range(length):
        row_series = df.iloc[counter]
        label_str = row_series["hold_out_y_true"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_true_list = processed_str.split(",")
        label_str = row_series["hold_out_y_pred"]
        processed_str = re.sub("\'|\[|\]", "", label_str)
        hold_out_y_pred_list = processed_str.split(",")
        # print(row_series["trained_dataset"])
        print(str(row_series["combination_trained"]))

        print(f"trained dataset: {row_series['trained_dataset']}")
        print(f"concatenated dataset: {'_'.join(str(row_series['combination_trained']).split('_')[8:10])}")
        print(f"predicted dataset: {'_'.join(str(row_series['combination_trained']).split('_')[-2:])}")
        # break
        output_df = analyze_predictions_automl_concat(hold_out_y_true = hold_out_y_true_list,
                                                    hold_out_y_pred = hold_out_y_pred_list,
                                                    concatenated_dataset = '_'.join(str(row_series['combination_trained']).split('_')[8:10]),
                                                    selected_system = row_series["automl_system"],
                                                    method = row_series["method"],
                                                    percentage = row_series["percentage"],
                                                    trained_dataset = row_series["trained_dataset"],
                                                    predicted_dataset = "_".join(str(row_series["combination_trained"]).split("_")[-2:]),
                                                    automl_system = row_series["automl_system"]
                                                    )
        automl_concat_flaml_df = pd.concat([automl_concat_flaml_df, output_df])

Streaming output truncated to the last 5000 lines.
concatenated_dataset='pc_07'
predicted_dataset='pc_02'

base_dataset_pc_05_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_03
trained dataset: pc_05
concatenated dataset: pc_07
predicted dataset: pc_03
trained_dataset='pc_05'
concatenated_dataset='pc_07'
predicted_dataset='pc_03'

base_dataset_pc_05_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_04
trained dataset: pc_05
concatenated dataset: pc_07
predicted dataset: pc_04
trained_dataset='pc_05'
concatenated_dataset='pc_07'
predicted_dataset='pc_04'

base_dataset_pc_05_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_06
trained dataset: pc_05
concatenated dataset: pc_07
predicted dataset: pc_06
trained_dataset='pc_05'
concatenated_dataset='pc_07'
predicted_dataset='pc_06'

base_dataset_pc_05_dataset_sampled_to_concat_pc_07_percentage_75_holdout_dataset_to_be_predicted_pc_07
trained datas

# Writing Results For Statistics

## Concat

In [ ]:
# for system, df in [("flaml", automl_concat_flaml_df), ("ag", automl_concat_ag_df), ("rf", automl_concat_rf_df)]:
for system, df in [("rf", automl_concat_rf_df)]:
    print(f"Writing concat results: {system=}..")
    df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/output/statistics/concat_{system}.csv", sep=";", index=False)

Writing concat results: system='rf'..


## HPO

In [8]:
for system, df in [("all_systems", automl_hpo_df)]:
    print(f"Writing HPO results: {system=}..")
    df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/output/statistics/hpo_{system}.csv", sep=";", index=False)

Writing HPO results: system='all_systems'..


## Sampled

In [ ]:
for system, df in [("all_systems", automl_sampled_df)]:
    print(f"Writing sampled results: {system=}..")
    df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/output/statistics/sampled_{system}.csv", sep=";", index=False)

Writing sampled results: system='all_systems'..


## Llama

In [ ]:
for system, df in [("llama", llama_output_results_df)]:
    print(f"Writing llama results: {system=}..")
    df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/output/statistics/{system}.csv", sep=";", index=False)

Writing llama results: system='llama'..


## Mistral

In [ ]:
for system, df in [("mistral", mistral_output_results_df)]:
    print(f"Writing mistral results: {system=}..")
    df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/output/statistics/{system}.csv", sep=";", index=False)

Writing mistral results: system='mistral'..


# Friedman Test

In [ ]:
automl_concat_ag_df["are_concat_pred_equal"] = automl_concat_ag_df.apply(
    lambda row: row.loc["concatenated_dataset"] == row.loc["predicted_dataset"], axis=1
)
automl_concat_ag_df = automl_concat_ag_df.sort_values(by=["trained_dataset"])
automl_concat_ag_df

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
0,ed_03,ed_01,ed_01,AutoGluon,gmm_random,25,"[disgust, anger, surprise]","[gratitude, admiration, neutral, grief, disapp...",False,0.307054,0.132780,"[sadness, anger, disgust, surprise, joy, fear]","[sadness, gratitude, admiration, neutral, grie...",True
0,ed_03,ed_03,ed_03,AutoGluon,gmm_random,75,"[pride, relief, grief, nervousness]",[],False,0.294867,0.456425,"[pride, gratitude, annoyance, disapproval, emb...","[gratitude, annoyance, disapproval, embarrassm...",True
0,ed_03,ed_02,ed_02,AutoGluon,gmm_random,75,[],"[disapproval, neutral, amusement, admiration]",False,0.342871,0.708556,"[sadness, anger, love, surprise, joy, fear]","[sadness, amusement, admiration, disapproval, ...",True
0,ed_03,ed_01,ed_01,AutoGluon,gmm_random,75,"[disgust, fear, surprise]","[neutral, curiosity, optimism, admiration]",False,0.307054,0.037344,"[sadness, anger, disgust, surprise, joy, fear]","[sadness, admiration, anger, neutral, curiosit...",True
0,ed_03,ed_03,ed_03,AutoGluon,gmm_random,50,[],[],False,0.294867,0.397618,"[pride, gratitude, annoyance, disapproval, emb...","[pride, gratitude, annoyance, disapproval, emb...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,ua_06,ua_06,ua_06,AutoGluon,gmm_random,50,[],[],False,0.442095,0.597105,"[7, 3, 1, 10, 5]","[7, 3, 1, 10, 5]",True
0,ua_06,ua_01,ua_01,AutoGluon,gmm_random,75,[],[],False,0.558380,0.598300,"[7, 3, 1, 10, 5]","[7, 3, 1, 10, 5]",True
0,ua_06,ua_03,ua_03,AutoGluon,gmm_random,75,[],[],False,0.666016,0.671220,"[7, 3, 1, 10, 5]","[7, 3, 1, 10, 5]",True
0,ua_06,ua_05,ua_05,AutoGluon,gmm_random,75,[],[],False,0.776119,0.621891,"[7, 3, 1, 10, 5]","[7, 3, 1, 10, 5]",True


In [ ]:
# drop all the datasets that used sd_02 so that it is possible to calculate considering all the methods --- or consider only the 41 datasets that appear in the concat case


# use everything based on the predicted dataset

# concat case:

# autogluon - separate for each percentage after
automl_concat_ag_df["are_concat_pred_equal"] = automl_concat_ag_df.apply(lambda row: row["concatenated_dataset"] == row["predicted_dataset"], axis=1)
equal_concat_ag_df = automl_concat_ag_df[automl_concat_ag_df["are_concat_pred_equal"]]
diff_concat_ag_df = automl_concat_ag_df[~(automl_concat_ag_df["are_concat_pred_equal"])]
grouped_equal_concat_ag_df = equal_concat_ag_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_equal_concat_ag_df = grouped_equal_concat_ag_df.reset_index()
grouped_equal_concat_ag_df = grouped_equal_concat_ag_df.rename(columns={"hold_out_f1_score": "hold_out_equal_ag_concat"})
grouped_equal_concat_ag_df_25 = grouped_equal_concat_ag_df[grouped_equal_concat_ag_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_ag_df_25 = grouped_equal_concat_ag_df.rename(columns={"hold_out_equal_ag_concat": "hold_out_equal_ag_concat_25"})
grouped_equal_concat_ag_df_50 = grouped_equal_concat_ag_df[grouped_equal_concat_ag_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_ag_df_50 = grouped_equal_concat_ag_df.rename(columns={"hold_out_equal_ag_concat": "hold_out_equal_ag_concat_50"})
grouped_equal_concat_ag_df_75 = grouped_equal_concat_ag_df[grouped_equal_concat_ag_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_ag_df_75 = grouped_equal_concat_ag_df.rename(columns={"hold_out_equal_ag_concat": "hold_out_equal_ag_concat_75"})
grouped_diff_concat_ag_df = diff_concat_ag_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_diff_concat_ag_df = grouped_diff_concat_ag_df.reset_index()
grouped_diff_concat_ag_df = grouped_diff_concat_ag_df.rename(columns={"hold_out_f1_score": "hold_out_diff_ag_concat"})
grouped_diff_concat_ag_df_25 = grouped_diff_concat_ag_df[grouped_diff_concat_ag_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_ag_df_25 = grouped_diff_concat_ag_df.rename(columns={"hold_out_diff_ag_concat": "hold_out_diff_ag_concat_25"})
grouped_diff_concat_ag_df_50 = grouped_diff_concat_ag_df[grouped_diff_concat_ag_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_ag_df_50 = grouped_diff_concat_ag_df.rename(columns={"hold_out_diff_ag_concat": "hold_out_diff_ag_concat_50"})
grouped_diff_concat_ag_df_75 = grouped_diff_concat_ag_df[grouped_diff_concat_ag_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_ag_df_75 = grouped_diff_concat_ag_df.rename(columns={"hold_out_diff_ag_concat": "hold_out_diff_ag_concat_75"})

# flaml
automl_concat_flaml_df["are_concat_pred_equal"] = automl_concat_flaml_df.apply(lambda row: row["concatenated_dataset"] == row["predicted_dataset"], axis=1)
equal_concat_flaml_df = automl_concat_flaml_df[automl_concat_flaml_df["are_concat_pred_equal"]]
diff_concat_flaml_df = automl_concat_flaml_df[~(automl_concat_flaml_df["are_concat_pred_equal"])]
grouped_equal_concat_flaml_df = equal_concat_flaml_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_equal_concat_flaml_df = grouped_equal_concat_flaml_df.reset_index()
grouped_equal_concat_flaml_df = grouped_equal_concat_flaml_df.rename(columns={"hold_out_f1_score": "hold_out_equal_flaml_concat"})
grouped_equal_concat_flaml_df_25 = grouped_equal_concat_flaml_df[grouped_equal_concat_flaml_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_flaml_df_25 = grouped_equal_concat_flaml_df.rename(columns={"hold_out_equal_flaml_concat": "hold_out_equal_flaml_concat_25"})
grouped_equal_concat_flaml_df_50 = grouped_equal_concat_flaml_df[grouped_equal_concat_flaml_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_flaml_df_50 = grouped_equal_concat_flaml_df.rename(columns={"hold_out_equal_flaml_concat": "hold_out_equal_flaml_concat_50"})
grouped_equal_concat_flaml_df_75 = grouped_equal_concat_flaml_df[grouped_equal_concat_flaml_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_flaml_df_75 = grouped_equal_concat_flaml_df.rename(columns={"hold_out_equal_flaml_concat": "hold_out_equal_flaml_concat_75"})
grouped_diff_concat_flaml_df = diff_concat_flaml_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_diff_concat_flaml_df = grouped_diff_concat_flaml_df.reset_index()
grouped_diff_concat_flaml_df = grouped_diff_concat_flaml_df.rename(columns={"hold_out_f1_score": "hold_out_diff_flaml_concat"})
grouped_diff_concat_flaml_df_25 = grouped_diff_concat_flaml_df[grouped_diff_concat_flaml_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_flaml_df_25 = grouped_diff_concat_flaml_df.rename(columns={"hold_out_diff_flaml_concat": "hold_out_diff_flaml_concat_25"})
grouped_diff_concat_flaml_df_50 = grouped_diff_concat_flaml_df[grouped_diff_concat_flaml_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_flaml_df_50 = grouped_diff_concat_flaml_df.rename(columns={"hold_out_diff_flaml_concat": "hold_out_diff_flaml_concat_50"})
grouped_diff_concat_flaml_df_75 = grouped_diff_concat_flaml_df[grouped_diff_concat_flaml_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_flaml_df_75 = grouped_diff_concat_flaml_df.rename(columns={"hold_out_diff_flaml_concat": "hold_out_diff_flaml_concat_75"})

# random forest
automl_concat_rf_df["are_concat_pred_equal"] = automl_concat_rf_df.apply(lambda row: row["concatenated_dataset"] == row["predicted_dataset"], axis=1)
equal_concat_rf_df = automl_concat_rf_df[automl_concat_rf_df["are_concat_pred_equal"]]
diff_concat_rf_df = automl_concat_rf_df[~(automl_concat_rf_df["are_concat_pred_equal"])]
grouped_equal_concat_rf_df = equal_concat_rf_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_equal_concat_rf_df = grouped_equal_concat_rf_df.reset_index()
grouped_equal_concat_rf_df = grouped_equal_concat_rf_df.rename(columns={"hold_out_f1_score": "hold_out_equal_rf_concat"})
grouped_equal_concat_rf_df_25 = grouped_equal_concat_rf_df[grouped_equal_concat_rf_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_rf_df_25 = grouped_equal_concat_rf_df.rename(columns={"hold_out_equal_rf_concat": "hold_out_equal_rf_concat_25"})
grouped_equal_concat_rf_df_50 = grouped_equal_concat_rf_df[grouped_equal_concat_rf_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_rf_df_50 = grouped_equal_concat_rf_df.rename(columns={"hold_out_equal_rf_concat": "hold_out_equal_rf_concat_50"})
grouped_equal_concat_rf_df_75 = grouped_equal_concat_rf_df[grouped_equal_concat_rf_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_equal_concat_rf_df_75 = grouped_equal_concat_rf_df.rename(columns={"hold_out_equal_rf_concat": "hold_out_equal_rf_concat_75"})
grouped_diff_concat_rf_df = diff_concat_rf_df.groupby(["predicted_dataset","automl_system", "method", "percentage"]).agg({"hold_out_f1_score": "mean"})
grouped_diff_concat_rf_df = grouped_diff_concat_rf_df.reset_index()
grouped_diff_concat_rf_df = grouped_diff_concat_rf_df.rename(columns={"hold_out_f1_score": "hold_out_diff_rf_concat"})
grouped_diff_concat_rf_df_25 = grouped_diff_concat_rf_df[grouped_diff_concat_rf_df["percentage"]==25][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_rf_df_25 = grouped_diff_concat_rf_df.rename(columns={"hold_out_diff_rf_concat": "hold_out_diff_rf_concat_25"})
grouped_diff_concat_rf_df_50 = grouped_diff_concat_rf_df[grouped_diff_concat_rf_df["percentage"]==50][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_rf_df_50 = grouped_diff_concat_rf_df.rename(columns={"hold_out_diff_rf_concat": "hold_out_diff_rf_concat_50"})
grouped_diff_concat_rf_df_75 = grouped_diff_concat_rf_df[grouped_diff_concat_rf_df["percentage"]==75][["predicted_dataset","hold_out_f1_score"]]
grouped_diff_concat_rf_df_75 = grouped_diff_concat_rf_df.rename(columns={"hold_out_diff_rf_concat": "hold_out_diff_rf_concat_75"})


# Llama
zero_llama_output_results_df = llama_output_results_df[llama_output_results_df["number_of_examples"]==0]
zero_llama_output_results_df = zero_llama_output_results_df[["predicted_dataset","hold_out_f1_score"]]
zero_llama_output_results_df = zero_llama_output_results_df.rename(columns={"hold_out_f1_score":"llama_zero_shot_hold_out_f1_score"})
five_llama_output_results_df = llama_output_results_df[llama_output_results_df["number_of_examples"]==5]
five_llama_output_results_df = five_llama_output_results_df[["predicted_dataset","hold_out_f1_score"]]
five_llama_output_results_df = five_llama_output_results_df.rename(columns={"hold_out_f1_score":"llama_five_shot_hold_out_f1_score"})


# Mistral
zero_mistral_output_results_df = mistral_output_results_df[mistral_output_results_df["number_of_examples"]==0]
zero_mistral_output_results_df = zero_mistral_output_results_df[["predicted_dataset","hold_out_f1_score"]]
zero_mistral_output_results_df = zero_mistral_output_results_df.rename(columns={"hold_out_f1_score":"mistral_zero_shot_hold_out_f1_score"})
five_mistral_output_results_df = mistral_output_results_df[mistral_output_results_df["number_of_examples"]==5]
five_mistral_output_results_df = five_mistral_output_results_df[["predicted_dataset","hold_out_f1_score"]]
five_mistral_output_results_df = five_mistral_output_results_df.rename(columns={"hold_out_f1_score":"mistral_five_shot_hold_out_f1_score"})



# HPO





KeyError: "['hold_out_f1_score'] not in index"

In [ ]:
# join based on predicted_dataset column

In [ ]:
results_concat_hs_list = os.listdir("/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs")
results_concat_hs_list = [f"/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/{item}" for item in results_concat_hs_list]
results_concat_hs_list.sort()
results_concat_hs_list

['/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_autogluon_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_25_holdout_dataset_to_be_predicted_hs_04.csv',
 '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_autogluon_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_25_holdout_dataset_to_be_predicted_hs_14.csv',
 '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_autogluon_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_50_holdout_dataset_to_be_predicted_hs_04.csv',
 '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_autogluon_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_50_holdout_dataset_to_be_predicted_hs_14.csv',
 '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_autogluon_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_75_holdout_datase

In [ ]:
results_concat_ag_hs_04_list = [item for item in results_concat_hs_list if "part_autogluon_summary_gmm_random_base_dataset_hs_04_" in item]
results_concat_ag_hs_14_list = [item for item in results_concat_hs_list if "part_autogluon_summary_gmm_random_base_dataset_hs_14_" in item]
results_concat_flaml_hs_04_list = [item for item in results_concat_hs_list if "part_flaml_100_combination_summary_gmm_random_base_dataset_hs_04_" in item]
results_concat_flaml_hs_14_list = [item for item in results_concat_hs_list if "part_flaml_100_combination_summary_gmm_random_base_dataset_hs_14_" in item]
print(f"{results_concat_flaml_hs_04_list=}")
print(f"{results_concat_flaml_hs_14_list=}")

results_concat_flaml_hs_04_list=['/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_flaml_100_combination_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_25_holdout_dataset_to_be_predicted_hs_04.csv', '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_flaml_100_combination_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_25_holdout_dataset_to_be_predicted_hs_14.csv', '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_flaml_100_combination_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_50_holdout_dataset_to_be_predicted_hs_04.csv', '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_flaml_100_combination_summary_gmm_random_base_dataset_hs_04_dataset_sampled_to_concat_hs_14_percentage_50_holdout_dataset_to_be_predicted_hs_14.csv', '/content/drive/MyDrive/Colab Notebooks/output/concatenate_hs/part_flaml_100_combination_summary_gm

In [ ]:
flaml_hs_04 = pd.DataFrame()
for file_name in results_concat_flaml_hs_04_list:
    df = pd.read_csv(file_name, sep=";")
    flaml_hs_04 = pd.concat([flaml_hs_04, df])
flaml_hs_04.to_csv("/content/drive/MyDrive/Colab Notebooks/output/concat/flaml/flaml_100_combination_summary_gmm_random_hs_04.csv", sep=";", index=False)
flaml_hs_04

,duration_training,duration_prediction,automl_system,dataset_compared,trained_dataset,hold_out_f1_score,validation_best_score,validation_all_scores,5_fold_validation_scores,hold_out_y_true,hold_out_y_pred,hold_out_y_pred_proba,name_best_classifier,method,percentage,combination_trained
0,1 days 00:02:58.629547,0 days 00:00:00.166805,FLAML,hs_04,hs_04,0.924859,0.050586,"[0.0953717564870259, 0.08202345309381243, 0.07...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 62.39085164291736},gmm_random,25,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 00:02:58.629547,0 days 00:00:00.025555,FLAML,hs_14,hs_04,0.600877,0.050586,"[0.0953717564870259, 0.08202345309381243, 0.07...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 62.39085164291736},gmm_random,25,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 00:03:27.480301,0 days 00:00:00.204950,FLAML,hs_04,hs_04,0.924761,0.052236,"[0.09725365465777724, 0.08110587803535385, 0.0...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 20.972940149254317},gmm_random,50,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 00:03:27.480301,0 days 00:00:00.035244,FLAML,hs_14,hs_04,0.605263,0.052236,"[0.09725365465777724, 0.08110587803535385, 0.0...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 20.972940149254317},gmm_random,50,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 06:14:54.178023,0 days 00:00:01.248665,FLAML,hs_04,hs_04,0.928077,0.052442,"[0.09930397215888631, 0.08166326653066125, 0.0...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]","xgb_limitdepth {'n_estimators': 8811, 'max_dep...",gmm_random,75,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 06:14:54.178023,0 days 00:00:00.164490,FLAML,hs_14,hs_04,0.894737,0.052442,"[0.09930397215888631, 0.08166326653066125, 0.0...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","toxic,not_toxic,not_toxic,toxic,not_toxic,not_...","[0, 0, 0, 0, 0]","xgb_limitdepth {'n_estimators': 8811, 'max_dep...",gmm_random,75,base_dataset_hs_04_dataset_sampled_to_concat_h...


In [ ]:
flaml_hs_14 = pd.DataFrame()
for file_name in results_concat_flaml_hs_14_list:
    df = pd.read_csv(file_name, sep=";")
    flaml_hs_14 = pd.concat([flaml_hs_14, df])
flaml_hs_14.to_csv("/content/drive/MyDrive/Colab Notebooks/output/concat/flaml/flaml_100_combination_summary_gmm_random_hs_14.csv", sep=";", index=False)
flaml_hs_14

,duration_training,duration_prediction,automl_system,dataset_compared,trained_dataset,hold_out_f1_score,validation_best_score,validation_all_scores,5_fold_validation_scores,hold_out_y_true,hold_out_y_pred,hold_out_y_pred_proba,name_best_classifier,method,percentage,combination_trained
0,1 days 00:00:57.231633,0 days 00:00:00.185942,FLAML,hs_04,hs_14,0.921835,0.078725,"[0.11459415788823835, 0.10458764475674956, 0.0...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 59.59447866374897},gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 00:00:57.231633,0 days 00:00:00.007501,FLAML,hs_14,hs_14,0.684211,0.078725,"[0.11459415788823835, 0.10458764475674956, 0.0...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",lrl1 {'C': 59.59447866374897},gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 00:02:35.665879,0 days 00:00:00.214608,FLAML,hs_04,hs_14,0.923184,0.065855,"[0.10420182783519005, 0.09261775995231866, 0.0...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 64.00000000000004},gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 00:02:35.665879,0 days 00:00:00.008493,FLAML,hs_14,hs_14,0.671053,0.065855,"[0.10420182783519005, 0.09261775995231866, 0.0...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",lrl1 {'C': 64.00000000000004},gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 00:02:27.176101,0 days 00:00:00.267799,FLAML,hs_04,hs_14,0.924712,0.060743,"[0.09957870547682879, 0.0868632707774799, 0.08...","[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",lrl1 {'C': 1.598949466649275},gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 00:02:27.176101,0 days 00:00:00.013415,FLAML,hs_14,hs_14,0.662281,0.060743,"[0.09957870547682879, 0.0868632707774799, 0.08...","[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",lrl1 {'C': 1.598949466649275},gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...


In [ ]:
# flaml_100_combination_summary_gmm_random_hs_04.csv
# autogluon_summary_gmm_random_hs_04.csv

ag_hs_04 = pd.DataFrame()
for file_name in results_concat_ag_hs_04_list:
    df = pd.read_csv(file_name, sep=";")
    ag_hs_04 = pd.concat([ag_hs_04, df])
ag_hs_04.to_csv("/content/drive/MyDrive/Colab Notebooks/output/concat/autogluon/autogluon_summary_gmm_random_hs_04.csv", sep=";", index=False)
ag_hs_04

,duration_training,duration_prediction,automl_system,dataset_compared,trained_dataset,hold_out_f1_score,validation_best_score,validation_all_scores,5_fold_validation_scores,hold_out_y_true,hold_out_y_pred,hold_out_y_pred_proba,name_best_classifier,method,percentage,combination_trained
0,1 days 03:31:32.979474,0 days 00:01:10.979153,AutoGluon,hs_04,hs_04,0.926451,0.950915,0.950915,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 12} {'us...,gmm_random,25,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 03:31:32.979474,0 days 00:00:00.844367,AutoGluon,hs_14,hs_04,0.653509,0.950915,0.950915,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 12} {'us...,gmm_random,25,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 04:20:39.407471,0 days 00:02:08.598250,AutoGluon,hs_04,hs_04,0.928288,0.948821,0.948821,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 14} {'us...,gmm_random,50,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,1 days 04:20:39.407471,0 days 00:00:00.832521,AutoGluon,hs_14,hs_04,0.653509,0.948821,0.948821,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 14} {'us...,gmm_random,50,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,0 days 23:26:30.847923,0 days 00:00:44.508574,AutoGluon,hs_04,hs_04,0.927167,0.946896,0.946896,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 81} {'us...,gmm_random,75,base_dataset_hs_04_dataset_sampled_to_concat_h...
0,0 days 23:26:30.847923,0 days 00:00:01.029912,AutoGluon,hs_14,hs_04,0.710526,0.946896,0.946896,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 81} {'us...,gmm_random,75,base_dataset_hs_04_dataset_sampled_to_concat_h...


In [ ]:
# flaml_100_combination_summary_gmm_random_hs_04.csv
# autogluon_summary_gmm_random_hs_04.csv

ag_hs_14 = pd.DataFrame()
for file_name in results_concat_ag_hs_14_list:
    df = pd.read_csv(file_name, sep=";")
    ag_hs_14 = pd.concat([ag_hs_14, df])
ag_hs_14.to_csv("/content/drive/MyDrive/Colab Notebooks/output/concat/autogluon/autogluon_summary_gmm_random_hs_14.csv", sep=";", index=False)
ag_hs_14

,duration_training,duration_prediction,automl_system,dataset_compared,trained_dataset,hold_out_f1_score,validation_best_score,validation_all_scores,5_fold_validation_scores,hold_out_y_true,hold_out_y_pred,hold_out_y_pred_proba,name_best_classifier,method,percentage,combination_trained
0,1 days 02:06:55.402750,0 days 00:02:09.854032,AutoGluon,hs_04,hs_14,0.914602,0.928088,0.928088,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 50} {'us...,gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 02:06:55.402750,0 days 00:00:01.106444,AutoGluon,hs_14,hs_14,0.820175,0.928088,0.928088,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","toxic,not_toxic,not_toxic,toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 50} {'us...,gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 01:05:24.618439,0 days 00:00:04.024540,AutoGluon,hs_04,hs_14,0.923770,0.938134,0.938134,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 45} {'us...,gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 01:05:24.618439,0 days 00:00:00.629935,AutoGluon,hs_14,hs_14,0.780702,0.938134,0.938134,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 45} {'us...,gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 03:40:09.179166,0 days 00:01:52.903063,AutoGluon,hs_04,hs_14,0.924143,0.942762,0.942762,"[0, 0, 0, 0, 0]","not_toxic,toxic,not_toxic,not_toxic,not_toxic,...","not_toxic,not_toxic,not_toxic,not_toxic,not_to...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 18} {'us...,gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...
0,1 days 03:40:09.179166,0 days 00:00:00.860056,AutoGluon,hs_14,hs_14,0.732456,0.942762,0.942762,"[0, 0, 0, 0, 0]","toxic,not_toxic,not_toxic,toxic,not_toxic,toxi...","not_toxic,not_toxic,not_toxic,toxic,not_toxic,...","[0, 0, 0, 0, 0]",WeightedEnsemble_L2 {'ensemble_size': 18} {'us...,gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...


In [ ]:
df = pd.read_csv(results_concat_hs_list[-1], sep=";")
df = df[df["trained_dataset"]=="hs_14"]
df["bool_combination_trained"] = df["combination_trained"].apply(lambda x: True if "base_dataset_hs_04_" not in x else False)
df = df[df["bool_combination_trained"]]
df = df.drop_duplicates()
df = df.drop(columns=["bool_combination_trained"])
df.to_csv(results_concat_hs_list[-1], sep=";", index_label=False)
df

,duration_training,duration_prediction,automl_system,dataset_compared,trained_dataset,hold_out_f1_score,validation_best_score,validation_all_scores,5_fold_validation_scores,hold_out_y_true,hold_out_y_pred,hold_out_y_pred_proba,name_best_classifier,method,percentage,combination_trained
0,0 days 00:00:32.865293,0 days 00:00:00.170089,Random Forest with Random Search,hs_04,hs_14,0.906004,0.885406,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['not_toxic,toxic,not_toxic,not_toxic,not_toxi...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
1,0 days 00:00:32.865293,0 days 00:00:00.022961,Random Forest with Random Search,hs_14,hs_14,0.517544,0.885406,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['toxic,not_toxic,not_toxic,toxic,not_toxic,to...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,25,base_dataset_hs_14_dataset_sampled_to_concat_h...
2,0 days 00:00:49.890387,0 days 00:00:00.210659,Random Forest with Random Search,hs_04,hs_14,0.906004,0.895798,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['not_toxic,toxic,not_toxic,not_toxic,not_toxi...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
3,0 days 00:00:49.890387,0 days 00:00:00.061869,Random Forest with Random Search,hs_14,hs_14,0.517544,0.895798,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['toxic,not_toxic,not_toxic,toxic,not_toxic,to...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,50,base_dataset_hs_14_dataset_sampled_to_concat_h...
4,0 days 00:01:06.270241,0 days 00:00:00.219920,Random Forest with Random Search,hs_04,hs_14,0.906004,0.900464,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['not_toxic,toxic,not_toxic,not_toxic,not_toxi...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...
5,0 days 00:01:06.270241,0 days 00:00:00.037418,Random Forest with Random Search,hs_14,hs_14,0.517544,0.900464,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","['toxic,not_toxic,not_toxic,toxic,not_toxic,to...","['not_toxic,not_toxic,not_toxic,not_toxic,not_...","[0, 0, 0, 0, 0]",Random Forest,gmm_random,75,base_dataset_hs_14_dataset_sampled_to_concat_h...


In [ ]:
df["combination_trained"].value_counts()

,count
combination_trained,
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_25_holdout_dataset_to_be_predicted_hs_04,1
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_25_holdout_dataset_to_be_predicted_hs_14,1
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_50_holdout_dataset_to_be_predicted_hs_04,1
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_50_holdout_dataset_to_be_predicted_hs_14,1
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_75_holdout_dataset_to_be_predicted_hs_04,1
base_dataset_hs_14_dataset_sampled_to_concat_hs_04_percentage_75_holdout_dataset_to_be_predicted_hs_14,1


In [ ]:
# df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/output/random_forest_hs_15/random_forest_100_hpo_extraction_summary_gmm_random_hs_15.csv",sep=";")
# df = df[(df["trained_dataset"]=="hs_15") & (df["dataset_compared"] == "hs_15")]
# columns_df = df.columns
# # df = df[columns_df[1:]]
# df = df.drop_duplicates(subset=columns_df[2:])
# df.to_csv("/content/drive/MyDrive/Colab Notebooks/output/random_forest_hs_15/random_forest_100_hpo_extraction_summary_gmm_random_hs_15.csv", sep=";", index_label=False, index=False)

In [ ]:
df["combination_trained"].value_counts()

,count
combination_trained,
base_dataset_hs_15_holdout_dataset_to_be_predicted_hs_15,3
